In [1]:
from scripts.python.simulateCommunityModel import simulateCommunityModel
from scripts.python.simulateCommunityModel_with_beta import simulateCommunityModel_with_beta
import pandas as pd

In [2]:
mu = 1.786 # growth rate
modelNames = ['Lachnospiraceae_species','Bacteroides_ovatus','Bifidobacterium_longum',
'Collinsella_aerofaciens','Escherichia_coli','Klebsiella_aerogenes','Muricomes_intestini','Parabacteroides_distasonis']
modelPaths = ['./Data/HDC/HDC1_gapseq/HDC_180_genome.xml',
'./Data/HDC/HDC1_gapseq/HDC_376_genome.xml',
'./Data/HDC/HDC1_gapseq/HDC_370_genome.xml',
'./Data/HDC/HDC1_gapseq/HDC_374_genome.xml',
'./Data/HDC/HDC1_gapseq/HDC_382_genome.xml',
'./Data/HDC/HDC1_gapseq/HDC_bin1.xml',
'./Data/HDC/HDC1_gapseq/HDC_380_genome.xml',
'./Data/HDC/HDC1_gapseq/HDC_378_genome.xml']
modelWeights = [0.000001,0.008976,0.000124,0.000007,0.000085,0.000121,0.000082,0.000426]

modelDetails = pd.DataFrame({'Name':modelNames,'Path':modelPaths,'weight':modelWeights})
modelDetails.head()

,Name,Path,weight
0,Lachnospiraceae_species,./Data/HDC/HDC1_gapseq/HDC_180_genome.xml,0.000001
1,Bacteroides_ovatus,./Data/HDC/HDC1_gapseq/HDC_376_genome.xml,0.008976
2,Bifidobacterium_longum,./Data/HDC/HDC1_gapseq/HDC_370_genome.xml,0.000124
3,Collinsella_aerofaciens,./Data/HDC/HDC1_gapseq/HDC_374_genome.xml,0.000007
4,Escherichia_coli,./Data/HDC/HDC1_gapseq/HDC_382_genome.xml,0.000085


In [3]:
# exchange reactions (from metabolomics data)
exchangeRate = pd.read_excel('./Data/HDC/HDC1_metabolomics_preliminary_formodelling2.xlsx')
exchangeRate['GF_conc'] = exchangeRate['GF_Mean (mmol per g)']
exchangeRate['Microbiome_conc'] = exchangeRate['HDC1_Mean (mmol per g)']
exchangeRate['GF_conc_sd'] = exchangeRate['GF_Standard deviation (mmol per g)']
exchangeRate['Microbiome_conc_sd'] = exchangeRate['HDC1_Standard deviation (mmol per g)']
exchangeRate['metabolite ID'] = exchangeRate['modelseed ID']
exchangeRate.head()



,Quantified compounds,modelseed ID,HMDB ID,Metabolite present in gapseq model exchange reactions?,GF_Mean (mmol per g),GF_Standard deviation (mmol per g),HDC1_Mean (mmol per g),HDC1_Standard deviation (mmol per g),Consumed or produced?,GF_conc,Microbiome_conc,GF_conc_sd,Microbiome_conc_sd,metabolite ID
0,Guanosine,cpd00311,HMDB0000133,Y,0.000523,0.000386,0.000284,0.000330,consumed,0.000523,0.000284,0.000386,0.000330,cpd00311
1,Argnine,cpd00051,HMDB0000517,Y,0.007678,0.006080,0.002062,0.001895,consumed,0.007678,0.002062,0.006080,0.001895,cpd00051
2,Asparagine,cpd00132,HMDB0000168,Y,0.007764,0.005528,0.000069,0.000080,consumed,0.007764,0.000069,0.005528,0.000080,cpd00132
3,Glutamine,cpd00053,HMDB0000641,Y,0.008064,0.005168,0.002055,0.001684,consumed,0.008064,0.002055,0.005168,0.001684,cpd00053
4,Glycine,cpd00033,HMDB0000123,Y,0.015165,0.011155,0.008150,0.006380,consumed,0.015165,0.008150,0.011155,0.006380,cpd00033


In [4]:
# loading the production rates
productionRate = pd.read_excel('./Data/HDC/HDC1_metabolomics_preliminary_formodelling2.xlsx',sheet_name='produced')
productionRate['GF_conc'] = productionRate['GF_Mean (mmol per g)']
productionRate['Microbiome_conc'] = productionRate['HDC1_Mean (mmol per g)']
productionRate['GF_conc_sd'] = productionRate['GF_Standard deviation (mmol per g)']
productionRate['Microbiome_conc_sd'] = productionRate['HDC1_Standard deviation (mmol per g)']
productionRate['metabolite ID'] = productionRate['modelseed ID']
productionRate.head()

,Quantified compounds,modelseed ID,HMDB ID,Metabolite present in gapseq model exchange reactions?,GF_Mean (mmol per g),GF_Standard deviation (mmol per g),HDC1_Mean (mmol per g),HDC1_Standard deviation (mmol per g),Consumed or produced?,GF_conc,Microbiome_conc,GF_conc_sd,Microbiome_conc_sd,metabolite ID
0,Adenosine,cpd00182,HMDB0000050,Y,1.410623e-05,0.000017,5.068257e-05,0.000053,produced,1.410623e-05,5.068257e-05,0.000017,0.000053,cpd00182
1,Cytidine,cpd00367,HMDB0000089,Y,3.014813e-04,0.000256,4.632410e-04,0.000503,produced,3.014813e-04,4.632410e-04,0.000256,0.000503,cpd00367
2,Cytosine,cpd00307,HMDB0000630,Y,5.700000e-07,0.000000,5.700000e-07,0.000000,produced,5.700000e-07,5.700000e-07,0.000000,0.000000,cpd00307
3,Deoxyguanosine,cpd00277,HMDB0000085,Y,1.591738e-04,0.000159,4.741676e-04,0.000571,produced,1.591738e-04,4.741676e-04,0.000159,0.000571,cpd00277
4,Deoxyuridine,cpd00412,HMDB0000012,Y,1.752750e-05,0.000016,4.067860e-04,0.000598,produced,1.752750e-05,4.067860e-04,0.000016,0.000598,cpd00412


## Simulating with the given SD

In [5]:
output_1 = simulateCommunityModel(modelDetails, exchangeRate, mu, solver='gurobi', productionRate=productionRate)

'180_genome' is not a valid SBML 'SId'.


Set parameter Username
Academic license - for non-commercial use only - expires 2026-02-21


'376_genome' is not a valid SBML 'SId'.
'370_genome' is not a valid SBML 'SId'.
'374_genome' is not a valid SBML 'SId'.
'382_genome' is not a valid SBML 'SId'.
'380_genome' is not a valid SBML 'SId'.
'378_genome' is not a valid SBML 'SId'.


Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpo1xvga89.lp
Reading time = 0.03 seconds
: 1454 rows, 3396 columns, 14966 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpx5fb3en5.lp
Reading time = 0.02 seconds
: 1661 rows, 4098 columns, 17964 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp_6rx05f7.lp
Reading time = 0.02 seconds
: 1355 rows, 3320 columns, 14560 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpkk45ey_n.lp
Reading time = 0.02 seconds
: 1177 rows, 2698 columns, 12056 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpwzktsl26.lp
Reading time = 0.03 seconds
: 2357 rows, 6008 columns, 26634 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp75lh7kbq.lp
Reading time = 0.03 seconds
: 2496 rows, 6336 columns, 28390 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp26a7in7x.lp
Reading time = 0.02 

c:\Users\Admin\anaconda3\envs\com_modelling\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Optimization is done
Comparing the actual and predicted biomass weights
                        Model   Predicted    Actual
0     Lachnospiraceae_species  143.869775  0.000001
1          Bacteroides_ovatus    1.000000  0.008976
2      Bifidobacterium_longum    0.000000  0.000124
3     Collinsella_aerofaciens   33.844141  0.000007
4            Escherichia_coli   -3.666882  0.000085
5        Klebsiella_aerogenes    0.016187  0.000121
6         Muricomes_intestini    1.000000  0.000082
7  Parabacteroides_distasonis -176.053400  0.000426
Comparing the actual and predicted metabolomics consumption rates by the community
   Metabolite    Predicted    Actual
0    cpd00311     0.000326  0.000426
1    cpd00051    48.958750  0.010031
2    cpd00132     0.003727  0.013742
3    cpd00053    -0.001504  0.010734
4    cpd00033  -848.891871  0.012528
5    cpd00119  -865.754580  0.002745
6    cpd00359 -4591.660586  0.000025
7    cpd00039  -438.154500  0.012356
8    cpd00054  3281.909583  0.008564
9    cp

c:\Users\Admin\anaconda3\envs\com_modelling\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
c:\Users\Admin\anaconda3\envs\com_modelling\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


# Increasing the bounds by taking 2 times the standard deviation

In [6]:
output_2 = simulateCommunityModel(modelDetails, exchangeRate, mu, solver='gurobi', n_sd =2, productionRate=productionRate)

'180_genome' is not a valid SBML 'SId'.
'376_genome' is not a valid SBML 'SId'.
'370_genome' is not a valid SBML 'SId'.
'374_genome' is not a valid SBML 'SId'.
'382_genome' is not a valid SBML 'SId'.
'380_genome' is not a valid SBML 'SId'.
'378_genome' is not a valid SBML 'SId'.


Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmppv_bgbrt.lp
Reading time = 0.02 seconds
: 1454 rows, 3396 columns, 14966 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpkjy8ah6a.lp
Reading time = 0.02 seconds
: 1661 rows, 4098 columns, 17964 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp398e_eb9.lp
Reading time = 0.02 seconds
: 1355 rows, 3320 columns, 14560 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp01rduuqx.lp
Reading time = 0.02 seconds
: 1177 rows, 2698 columns, 12056 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpq63y8mt7.lp
Reading time = 0.03 seconds
: 2357 rows, 6008 columns, 26634 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp0prmficc.lp
Reading time = 0.03 seconds
: 2496 rows, 6336 columns, 28390 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpzi8v_ffu.lp
Reading time = 0.02 

In [ ]:
# setting a relaxation on the total biomass constraint
output_2 = simulateCommunityModel(modelDetails, exchangeRate, mu, solver='gurobi', n_sd =2, productionRate=productionRate,alpha=100)

'180_genome' is not a valid SBML 'SId'.
'376_genome' is not a valid SBML 'SId'.
'370_genome' is not a valid SBML 'SId'.
'374_genome' is not a valid SBML 'SId'.
'382_genome' is not a valid SBML 'SId'.
'380_genome' is not a valid SBML 'SId'.
'378_genome' is not a valid SBML 'SId'.


Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpo6daodye.lp
Reading time = 0.02 seconds
: 1454 rows, 3396 columns, 14966 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpp8ihypt7.lp
Reading time = 0.02 seconds
: 1661 rows, 4098 columns, 17964 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpt4f0s4g8.lp
Reading time = 0.02 seconds
: 1355 rows, 3320 columns, 14560 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmplooe2vuo.lp
Reading time = 0.01 seconds
: 1177 rows, 2698 columns, 12056 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpj5lpdvxu.lp
Reading time = 0.03 seconds
: 2357 rows, 6008 columns, 26634 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpd7dnprdd.lp
Reading time = 0.03 seconds
: 2496 rows, 6336 columns, 28390 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpz3mv1lxs.lp
Reading time = 0.02 

# Simulating the community model by adding a beta variable to the metabolomics constraint. Also, the objective function will have a term for summation of all betas.

In [7]:
output_3 = simulateCommunityModel_with_beta(modelDetails, exchangeRate, mu, solver='gurobi', productionRate=productionRate)

'180_genome' is not a valid SBML 'SId'.
'376_genome' is not a valid SBML 'SId'.
'370_genome' is not a valid SBML 'SId'.
'374_genome' is not a valid SBML 'SId'.
'382_genome' is not a valid SBML 'SId'.
'380_genome' is not a valid SBML 'SId'.
'378_genome' is not a valid SBML 'SId'.


Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpur9vsag_.lp
Reading time = 0.02 seconds
: 1454 rows, 3396 columns, 14966 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpv2510z3r.lp
Reading time = 0.02 seconds
: 1661 rows, 4098 columns, 17964 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpi0_t0nbn.lp
Reading time = 0.02 seconds
: 1355 rows, 3320 columns, 14560 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpd8sd8w75.lp
Reading time = 0.02 seconds
: 1177 rows, 2698 columns, 12056 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpj21ot5vw.lp
Reading time = 0.03 seconds
: 2357 rows, 6008 columns, 26634 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmptgb2urh4.lp
Reading time = 0.03 seconds
: 2496 rows, 6336 columns, 28390 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmps1nou0tm.lp
Reading time = 0.02 

In [7]:
output_3 = simulateCommunityModel_with_beta(modelDetails, exchangeRate, 2, solver='gurobi', productionRate=productionRate,tradeoff=0.5)

'180_genome' is not a valid SBML 'SId'.
'376_genome' is not a valid SBML 'SId'.
'370_genome' is not a valid SBML 'SId'.
'374_genome' is not a valid SBML 'SId'.
'382_genome' is not a valid SBML 'SId'.
'380_genome' is not a valid SBML 'SId'.
'378_genome' is not a valid SBML 'SId'.


Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp3xghoox7.lp
Reading time = 0.02 seconds
: 1454 rows, 3396 columns, 14966 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpls9376cd.lp
Reading time = 0.02 seconds
: 1661 rows, 4098 columns, 17964 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp676uy4jk.lp
Reading time = 0.02 seconds
: 1355 rows, 3320 columns, 14560 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmp0eeb4uka.lp
Reading time = 0.02 seconds
: 1177 rows, 2698 columns, 12056 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpuungzehu.lp
Reading time = 0.03 seconds
: 2357 rows, 6008 columns, 26634 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpq9ml5fr4.lp
Reading time = 0.03 seconds
: 2496 rows, 6336 columns, 28390 nonzeros
Read LP format model from file C:\Users\Admin\AppData\Local\Temp\tmpp1r4gmm6.lp
Reading time = 0.02 